In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from google.colab import files

# Define base URL and target endpoint
BASE_URL = "https://investors.metals.co"
START_URL = f"{BASE_URL}/news-events/press-releases"

# Mimic a real browser to bypass Akamai bot detection
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

def scrape_press_releases():
    all_releases = []
    page = 0

    while True:
        # Construct the URL for the current page
        url = f"{START_URL}?page={page}"
        print(f"Fetching page {page + 1}...")

        response = requests.get(url, headers=HEADERS)

        if response.status_code != 200:
            print(f"Stopping. Hit a snag or reached the end. Status code: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all press release container modules based on the HTML structure
        release_items = soup.find_all('div', class_='et_pb_toggle_item')

        # If no elements are returned, we have reached past the final page
        if not release_items:
            print("No more press releases found. Finalizing dataset...")
            break

        for item in release_items:
            # 1. Extract Title and URL
            title_tag = item.find('h5', class_='et_pb_toggle_title')
            if title_tag:
                a_tag = title_tag.find('a')
                if a_tag:
                    title = a_tag.text.strip()
                    relative_url = a_tag.get('href', '')
                    # Reconstruct absolute URL if necessary
                    full_url = BASE_URL + relative_url if relative_url.startswith('/') else relative_url
                else:
                    continue
            else:
                continue

            # 2. Extract Date
            date_tag = item.find('span', class_='news-date')
            date = date_tag.text.strip() if date_tag else "N/A"

            # Append the extracted row to our master list
            all_releases.append({
                "Title": title,
                "Date": date,
                "URL": full_url
            })

        # Rate-limiting: Rest 1.5 seconds between pages to respect the server
        time.sleep(1.5)
        page += 1

    # Create DataFrame and export
    if all_releases:
        df = pd.DataFrame(all_releases)
        print(f"\nSuccess! Total items scraped: {len(df)}")

        # Save to CSV
        csv_filename = "metals_co_press_releases.csv"
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"Data saved locally to '{csv_filename}'. Downloading now...")

        # Trigger browser download within Colab
        files.download(csv_filename)
    else:
        print("No data was collected. Check your connection or block status.")

# Run the scraper
scrape_press_releases()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from google.colab import files

# File naming configurations
INPUT_CSV = "metals_co_press_releases.csv"
OUTPUT_CSV = "metals_co_press_releases_with_content.csv"

# Maintain browser headers to navigate Akamai protections safely
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def scrape_individual_content():
    # 1. Verify if the previous step's CSV exists
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Could not find '{INPUT_CSV}'. Please run the first code cell to generate it.")
        return

    # Load initial dataset
    df = pd.read_csv(INPUT_CSV)
    total_rows = len(df)
    print(f"Loaded {total_rows} URLs from index file. Starting deep crawl...\n")

    full_contents = []

    # 2. Iterate through each press release URL
    for index, row in df.iterrows():
        url = row['URL']
        print(f"[{index + 1}/{total_rows}] Scraping text from: {url}")

        try:
            response = requests.get(url, headers=HEADERS, timeout=15)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                # Targeted selectors matching this specific Drupal 10/NIR framework layout
                content_area = (
                    soup.find('div', class_='field--name-field-nir-news-body') or
                    soup.find('div', class_='field--name-body') or
                    soup.find('article') or
                    soup.find('div', id='ndq-content')
                )

                if content_area:
                    # Extracts text cleanly, keeping linebreaks between paragraphs intact
                    text_content = content_area.get_text(separator="\n").strip()
                else:
                    text_content = "Content block container not identified."
            else:
                text_content = f"Failed to fetch content. HTTP status: {response.status_code}"

        except Exception as e:
            text_content = f"Error occurred during extraction: {str(e)}"
            print(f"   Skipping or failing softly on row {index + 1}: {e}")

        # Store content temporarily
        full_contents.append(text_content)

        # Polite delay to prevent connection resets/IP throttling by Akamai
        time.sleep(1.5)

    # 3. Append content to dataset structure and save
    df['Full Content'] = full_contents

    # Reordering columns cleanly
    df = df[['URL', 'Title', 'Date', 'Full Content']]
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

    print(f"\nProcessing complete! Enriched dataset exported to '{OUTPUT_CSV}'. Triggering download...")
    files.download(OUTPUT_CSV)

# Run the deep text crawler
scrape_individual_content()